# 01. Corpus & Ground Truth

Notebook ini menyiapkan semua data untuk sistem retrieval:

| Langkah | Hasil (folder `data/`) |
|---|---|
| 1. Corpus Al-Qur'an | `quran.csv`: 6.236 ayat (Arab + Inggris) |
| 2. Corpus hadis | `hadith.csv`: 5 kitab hadis (Arab + Inggris) |
| 3. Corpus islamqa | `islamqa.csv`: tanya-jawab islamqa |
| 4. Ground truth pertanyaan | `qrels_question.csv`: 1 dokumen islamqa per query |
| 5. Ground truth ayat | `qrels_quran.csv`: ayat yang dikutip di jawaban islamqa |
| 6. Ground truth hadis | `qrels_hadith.csv`: hadis yang dikutip di jawaban islamqa |
| 7. Ringkasan & cek manual | statistik dan contoh untuk dicek |

Struktur folder yang dipakai:
```
final_project_2/
├── alquran/        en.sahih.json, quran-uthmani.json
├── hadist/         <folder per kitab>/eng-*.min.json, ara-*.min.json
├── notebooks/      notebook ini
└── data/           hasil (dibuat otomatis)
```

## 0. Setup

Atur lokasi file. `ISLAMQA_DIR` adalah folder data islamqa dari tugas sebelumnya.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

BASE        = Path("..")                                        # folder final_project_2
QURAN_DIR   = BASE / "alquran"
HADITH_DIR  = BASE / "hadist"
ISLAMQA_DIR = Path("../../information_retrieval")               # data islamqa tugas sebelumnya
DATA_OUT    = BASE / "data"
DATA_OUT.mkdir(exist_ok=True)

print("Folder hasil:", DATA_OUT.resolve())

Folder hasil: E:\Tugas Semester 7\NLP\chatbot\final_project_2\data


## 1. Corpus Al-Qur'an

Dua file digabung berdasarkan nomor surah dan ayat:
- `quran-uthmani.json`: teks Arab (untuk ditampilkan)
- `en.sahih.json`: terjemahan Inggris Saheeh International (untuk retrieval)

`doc_id` setiap ayat ditulis `surah:ayat`, misalnya `2:255`.

In [2]:
def load_quran(path):
    data = json.load(open(path, encoding="utf-8"))["data"]
    rows = []
    for surah in data["surahs"]:
        for ayah in surah["ayahs"]:
            rows.append({"surah": surah["number"], "ayah": ayah["numberInSurah"],
                         "surah_name": surah["englishName"],
                         "text": ayah["text"].replace("﻿", "").strip()})   # buang karakter BOM tak terlihat
    return pd.DataFrame(rows)

quran_en = load_quran(QURAN_DIR / "en.sahih.json")
quran_ar = load_quran(QURAN_DIR / "quran-uthmani.json")

quran = quran_en.rename(columns={"text": "text_en"}).merge(
    quran_ar[["surah", "ayah", "text"]].rename(columns={"text": "text_ar"}), on=["surah", "ayah"])
quran.insert(0, "doc_id", quran["surah"].astype(str) + ":" + quran["ayah"].astype(str))
quran = quran[["doc_id", "surah", "ayah", "surah_name", "text_ar", "text_en"]]

quran.to_csv(DATA_OUT / "quran.csv", index=False)
print(f"Jumlah ayat: {len(quran)}")
quran.head()

Jumlah ayat: 6236


,doc_id,surah,ayah,surah_name,text_ar,text_en
0,1:1,1,1,Al-Faatiha,بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"In the name of Allah, the Entirely Merciful, t..."
1,1:2,1,2,Al-Faatiha,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,"[All] praise is [due] to Allah, Lord of the wo..."
2,1:3,1,3,Al-Faatiha,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,"The Entirely Merciful, the Especially Merciful,"
3,1:4,1,4,Al-Faatiha,مَٰلِكِ يَوْمِ ٱلدِّينِ,Sovereign of the Day of Recompense.
4,1:5,1,5,Al-Faatiha,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,It is You we worship and You we ask for help.


## 2. Corpus hadis

Setiap folder kitab berisi file Inggris (`eng-*.min.json`) dan Arab (`ara-*.min.json`).
Keduanya digabung berdasarkan nomor hadis.

- `doc_id` ditulis `kitab:nomor`, misalnya `bukhari:6018`
- Beberapa nomor punya lebih dari satu riwayat (sub-nomor), teksnya digabung menjadi satu dokumen
- Hadis tanpa teks Inggris dibuang
- Sahih Muslim belum dimasukkan karena penomorannya berbeda dengan kutipan islamqa

In [3]:
def load_hadith(path):
    data = json.load(open(path, encoding="utf-8"))
    rows = []
    for h in data["hadiths"]:
        grade = "; ".join(f"{g['name']}: {g['grade']}" for g in h.get("grades", []))
        rows.append({"number": int(float(h["hadithnumber"])), "text": h["text"].strip(), "grade": grade})
    df = pd.DataFrame(rows)
    # gabungkan sub-nomor (mis. 1 dan 1.1) menjadi satu dokumen per nomor
    return df.groupby("number", as_index=False).agg(
        {"text": lambda t: "\n".join(x for x in t if x), "grade": "first"})

hadith_parts = []
for eng_file in sorted(HADITH_DIR.glob("*/eng-*.min.json")):
    book = eng_file.name.replace("eng-", "").replace(".min.json", "")      # mis. "bukhari"
    ara_file = eng_file.with_name(f"ara-{book}.min.json")
    en = load_hadith(eng_file).rename(columns={"text": "text_en"})
    ar = load_hadith(ara_file).rename(columns={"text": "text_ar"})
    df = en.merge(ar[["number", "text_ar"]], on="number", how="left")
    df.insert(0, "book", book)
    hadith_parts.append(df)
    print(f"{book:<10} {len(df):>5} nomor hadis")

hadith = pd.concat(hadith_parts, ignore_index=True)
hadith = hadith[hadith["text_en"] != ""]                                  # buang yang kosong
hadith.insert(0, "doc_id", hadith["book"] + ":" + hadith["number"].astype(str))
hadith = hadith[["doc_id", "book", "number", "grade", "text_ar", "text_en"]]

hadith.to_csv(DATA_OUT / "hadith.csv", index=False)
print(f"\nTotal hadis (berisi teks Inggris): {len(hadith):,}")
hadith.head()

bukhari     7563 nomor hadis
tirmidhi    3956 nomor hadis
abudawud    5274 nomor hadis
nasai       5758 nomor hadis
ibnmajah    4341 nomor hadis

Total hadis (berisi teks Inggris): 26,725


,doc_id,book,number,grade,text_ar,text_en
0,bukhari:1,bukhari,1,,حَدَّثَنَا الْحُمَيْدِيُّ عَبْدُ اللَّهِ بْنُ ...,Narrated 'Umar bin Al-Khattab: I heard Allah's...
1,bukhari:2,bukhari,2,,حَدَّثَنَا عَبْدُ اللَّهِ بْنُ يُوسُفَ، قَالَ ...,Narrated 'Aisha: (the mother of the faithful b...
2,bukhari:3,bukhari,3,,حَدَّثَنَا يَحْيَى بْنُ بُكَيْرٍ، قَالَ حَدَّث...,Narrated 'Aisha (the mother of the faithful be...
3,bukhari:4,bukhari,4,,قَالَ ابْنُ شِهَابٍ وَأَخْبَرَنِي أَبُو سَلَمَ...,Narrated Jabir bin 'Abdullah Al-Ansari (while ...
4,bukhari:5,bukhari,5,,حَدَّثَنَا مُوسَى بْنُ إِسْمَاعِيلَ، قَالَ حَد...,Narrated Said bin Jubair: Ibn 'Abbas in the ex...


## 3. Corpus islamqa

Data tanya-jawab islamqa dari tugas sebelumnya. `doc_id` = ID pertanyaan islamqa (sama dengan di URL).

In [4]:
islamqa = pd.read_csv(ISLAMQA_DIR / "islamqa_adab_akhlak_en_final.csv")
islamqa = islamqa.rename(columns={"id": "doc_id"})
islamqa["doc_id"] = islamqa["doc_id"].astype(str)
islamqa = islamqa[["doc_id", "url", "kategori", "judul", "pertanyaan", "jawaban"]]

islamqa.to_csv(DATA_OUT / "islamqa.csv", index=False)
print(f"Jumlah tanya-jawab: {len(islamqa):,}")
islamqa.head(3)

Jumlah tanya-jawab: 2,065


,doc_id,url,kategori,judul,pertanyaan,jawaban
0,577803,https://islamqa.info/en/answers/577803,Character & Heart-Softeners,Are there any deeds that will help to bring re...,"We were financially very well off, then sudden...",We ask Allah to relieve you of your worries an...
1,440985,https://islamqa.info/en/answers/440985,Character & Heart-Softeners,Are good deeds and bad deeds multiplied during...,Is sin multiplied during Rajab and the other s...,Table Of Contents What are the sacred months? ...
2,491457,https://islamqa.info/en/answers/491457,Character & Heart-Softeners,Should he tell his friend’s family about his m...,I have a friend whose mother likes me and she ...,Islam urges the Muslim to look for righteous f...


## 4. Ground truth: Retrieve Pertanyaan

Dari `soal_evaluasi.csv` tugas sebelumnya: 50 soal (25 asli + 25 parafrase),
masing-masing punya **1 dokumen kunci** (pertanyaan islamqa yang benar).

In [5]:
soal = pd.read_csv(ISLAMQA_DIR / "soal_evaluasi.csv")

qrels_question = pd.DataFrame({
    "query_id": [f"Q{i + 1:02d}" for i in range(len(soal))],
    "jenis":    soal["jenis"],
    "query":    soal["pertanyaan"],
    "doc_id":   soal["id_dokumen_kunci"].astype(str),
})
qrels_question.to_csv(DATA_OUT / "qrels_question.csv", index=False)

print(qrels_question["jenis"].value_counts().to_string())
print("Semua dokumen kunci ada di corpus islamqa:", qrels_question["doc_id"].isin(islamqa["doc_id"]).all())
qrels_question.head()

jenis
asli         25
parafrase    25
Semua dokumen kunci ada di corpus islamqa: True


,query_id,jenis,query,doc_id
0,Q01,asli,Ruling on using a tasbeeh ring,198080
1,Q02,asli,Meaning of Hasad,105471
2,Q03,asli,He used to take money without his parents’ kno...,45670
3,Q04,asli,Will a woman be rewarded for her work in the h...,69960
4,Q05,asli,A calamity befell him and he apostatized from ...,91580


## 5. Ground truth: Retrieve Al-Qur'an

Jawaban islamqa mengutip ayat di dalam kurung siku dengan beberapa variasi format:

| Format di jawaban | Ayat yang diambil |
|---|---|
| `[Al-Baqarah 2:216]` | 2:216 |
| `[Al-A'raf 7:55-56]` (rentang) | 7:55, 7:56 |
| `[An-Nur 24:37, 38]` (daftar) | 24:37, 24:38 |
| `[al-Mutaffifeen 83:14 – interpretation of the meaning]` | 83:14 |
| `[Ibrahim 14:5; Luqman 31:31]` (beberapa referensi) | 14:5, 31:31 |

- **Query** = pertanyaan islamqa (judul + isi pertanyaan), **bukan** jawabannya
- **Ground truth** = semua ayat yang dikutip di jawaban pertanyaan itu (bisa lebih dari satu)

In [6]:
# 1) ambil isi setiap kurung siku
QURAN_CITE = re.compile(r"\[([^\]]{3,200})\]")
# 2) di dalamnya cari referensi "Nama Surah surah:ayat", ayat bisa "14", "55-56", atau "37, 38"
VERSE_REF = re.compile(r"[A-Za-z'’`][A-Za-z'’`\- ]*?\s+(\d{1,3})\s*:\s*"
                       r"(\d{1,3}(?:\s*[-–]\s*\d{1,3})?(?:\s*,\s*\d{1,3}(?:\s*[-–]\s*\d{1,3})?)*)")

def extract_quran(answer):
    verses = []
    for bracket in QURAN_CITE.findall(answer):
        for surah, ayah_list in VERSE_REF.findall(bracket):   # satu kurung bisa berisi beberapa referensi
            for part in ayah_list.split(","):                   # "37, 38" -> ["37", " 38"]
                nums = [int(n) for n in re.findall(r"\d+", part)]
                if len(nums) == 1:
                    verses.append(f"{int(surah)}:{nums[0]}")
                elif 0 <= nums[1] - nums[0] <= 30:              # rentang "55-56"
                    verses += [f"{int(surah)}:{a}" for a in range(nums[0], nums[1] + 1)]
    return verses

# cek cepat
print(extract_quran("[Al-A'raf 7:55-56] [An-Nur 24:37, 38] [al-Mutaffifeen 83:14 – interpretation of the meaning]"))
print(extract_quran("[Ibrahim 14:5; Luqman 31:31; al-Shura 42:33]"))

rows = []
for _, r in islamqa.iterrows():
    for verse in dict.fromkeys(extract_quran(r["jawaban"])):             # tanpa duplikat
        rows.append({"query_id": r["doc_id"], "doc_id": verse})
qrels_quran = pd.DataFrame(rows)

# buang kutipan yang tidak ada di corpus (mis. salah ketik nomor ayat)
valid = qrels_quran["doc_id"].isin(quran["doc_id"])
print(f"Kutipan tidak valid dibuang: {(~valid).sum()}")
qrels_quran = qrels_quran[valid].reset_index(drop=True)

qrels_quran.to_csv(DATA_OUT / "qrels_quran.csv", index=False)
print(f"Pasangan query-ayat: {len(qrels_quran):,} | query yang punya ground truth: {qrels_quran['query_id'].nunique():,}")
qrels_quran.head()

['7:55', '7:56', '24:37', '24:38', '83:14']
['14:5', '31:31', '42:33']
Kutipan tidak valid dibuang: 1
Pasangan query-ayat: 3,228 | query yang punya ground truth: 947


,query_id,doc_id
0,577803,2:216
1,577803,39:10
2,577803,2:186
3,577803,6:43
4,577803,6:17


## 6. Ground truth: Retrieve Hadis

Jawaban islamqa mengutip hadis dengan format seperti:
`Narrated by al-Bukhari (6018)`, `at-Tirmidhi, 2036`, `Abu Dawood (4930)`.

Aturan ekstraksi:
- Nama kitab harus diikuti langsung nomor hadis, `(nomor)` atau `, nomor`
- Format jilid/halaman seperti `6/246` tidak diambil (bukan nomor hadis)
- Kutipan dari kitab al-Albani (mis. `Saheeh at-Tirmidhi (1234)`) tidak diambil, karena penomorannya berbeda.
  Pengecualian: `Saheeh al-Bukhari` tetap diambil karena itu memang nama kitab Bukhari
- Hanya nomor yang ada di corpus hadis yang dipakai

**Skor kecocokan teks.** Penomoran hadis di islamqa kadang berasal dari edisi lain, sehingga nomornya
menunjuk ke hadis yang salah di corpus kita. Untuk menandainya, teks ±500 karakter sebelum kutipan
(biasanya isi hadis yang dikutip islamqa) dibandingkan dengan teks hadis di corpus:

- `skor_teks` = proporsi kata penting dari kutipan islamqa yang juga ada di teks hadis corpus
- `terverifikasi` = `True` jika `skor_teks >= 0.1` (hadis acak biasanya hanya 0,00–0,04)

Skor rendah belum tentu salah: kadang islamqa hanya menyebut nomor tanpa mengutip isi hadisnya.
Karena itu tidak ada yang dibuang; evaluasi nanti bisa memakai **semua** ground truth atau **yang terverifikasi** saja.

In [7]:
BOOK_NAMES = {
    "bukhari":  r"(?:al-)?Bukhaa?ri",
    "tirmidhi": r"(?:at-)?Tirmidhi",
    "abudawud": r"Abu Daa?wood|Abu Dawud",
    "ibnmajah": r"Ibn Maa?jah",
    "nasai":    r"(?:an-)?Nasaa?[’'`]?i",
}

def extract_hadith(answer):
    """Kembalikan [(doc_id, posisi kutipan di jawaban), ...]."""
    found = []
    for book, name in BOOK_NAMES.items():
        pattern = rf"\b(?:{name})\s*(?:\(\s*(\d{{1,5}})\s*\)|,\s*(\d{{1,5}})\b(?!\s*/))"
        for m in re.finditer(pattern, answer):
            before = answer[max(0, m.start() - 15):m.start()].lower()
            # "Saheeh/Da'eef at-Tirmidhi (n)" = kitab al-Albani (nomor berbeda) -> lewati.
            # "Saheeh al-Bukhari (n)" tetap dipakai karena itu memang kitab Bukhari.
            if book != "bukhari" and re.search(r"(sahih|saheeh|da[`'’‘]?e?e?if)\s*$", before):
                continue
            found.append((f"{book}:{int(m.group(1) or m.group(2))}", m.start()))
    return found

STOPWORDS = set("""the a an and of to in is that he she it was for on with his her him they them be by as at this
from or not have has had you your i we our my me who which what when will would all are were said one there their
so if do did upon allah messenger prophet peace blessings narrated reported""".split())

def important_words(text):
    return {w for w in re.findall(r"[a-z]+", str(text).lower()) if len(w) > 2 and w not in STOPWORDS}

def text_score(answer, position, hadith_text):
    quoted = important_words(answer[max(0, position - 500):position])   # teks sebelum kutipan
    return len(quoted & important_words(hadith_text)) / len(quoted) if quoted else 0.0

hadith_text = hadith.set_index("doc_id")["text_en"]
rows = []
for _, r in islamqa.iterrows():
    for hadith_id, pos in extract_hadith(r["jawaban"]):
        if hadith_id not in hadith_text.index:                         # nomor tidak ada di corpus
            continue
        rows.append({"query_id": r["doc_id"], "doc_id": hadith_id,
                     "skor_teks": round(text_score(r["jawaban"], pos, hadith_text[hadith_id]), 3)})

# hadis yang sama bisa dikutip dua kali di satu jawaban: ambil skor tertinggi
qrels_hadith = pd.DataFrame(rows).groupby(["query_id", "doc_id"], as_index=False)["skor_teks"].max()
qrels_hadith["terverifikasi"] = qrels_hadith["skor_teks"] >= 0.1

qrels_hadith.to_csv(DATA_OUT / "qrels_hadith.csv", index=False)
print(f"Pasangan query-hadis: {len(qrels_hadith):,} | query yang punya ground truth: {qrels_hadith['query_id'].nunique():,}")
print(f"Terverifikasi: {qrels_hadith['terverifikasi'].sum():,} pasangan "
      f"({qrels_hadith['terverifikasi'].mean():.0%}) dari {qrels_hadith[qrels_hadith['terverifikasi']]['query_id'].nunique():,} query\n")
print(qrels_hadith.assign(kitab=qrels_hadith["doc_id"].str.split(":").str[0])
      .groupby("kitab").agg(pasangan=("doc_id", "size"), terverifikasi=("terverifikasi", "mean")).round(2).to_string())

Pasangan query-hadis: 2,008 | query yang punya ground truth: 898
Terverifikasi: 1,183 pasangan (59%) dari 628 query

          pasangan  terverifikasi
kitab                            
abudawud       316           0.68
bukhari       1007           0.52
ibnmajah       158           0.72
nasai           82           0.66
tirmidhi       445           0.63


## 7. Ringkasan & cek manual

Jumlah ground truth per query (penting untuk membaca Precision dan Recall nanti),
lalu beberapa contoh untuk dicek manual: apakah ayat/hadis yang diekstrak memang sesuai kutipan di jawaban.

In [8]:
def gt_summary(qrels, name):
    per_query = qrels.groupby("query_id").size()
    return {"ground truth": name, "query": per_query.size, "total pasangan": len(qrels),
            "rata-rata per query": round(per_query.mean(), 2), "median": per_query.median(),
            "min": per_query.min(), "maks": per_query.max()}

summary = pd.DataFrame([
    gt_summary(qrels_question, "pertanyaan islamqa"),
    gt_summary(qrels_quran, "ayat Al-Qur'an"),
    gt_summary(qrels_hadith, "hadis (semua)"),
    gt_summary(qrels_hadith[qrels_hadith["terverifikasi"]], "hadis (terverifikasi)"),
])
summary

,ground truth,query,total pasangan,rata-rata per query,median,min,maks
0,pertanyaan islamqa,50,50,1.00,1.0,1,1
1,ayat Al-Qur'an,947,3228,3.41,2.0,1,27
2,hadis (semua),898,2008,2.24,2.0,1,23
3,hadis (terverifikasi),628,1183,1.88,1.0,1,18


In [9]:
# Contoh untuk dicek manual: jalankan ulang untuk melihat contoh lain
sample_id = qrels_quran["query_id"].drop_duplicates().sample(1).iloc[0]
q = islamqa.set_index("doc_id").loc[sample_id]

print(f"ISLAMQA {sample_id}: {q['judul']}\n{q['url']}\n")
print("Kutipan ayat di jawaban:", [b for b in QURAN_CITE.findall(q["jawaban"]) if VERSE_REF.search(b)])
print("\nGround truth ayat:")
for verse in qrels_quran.loc[qrels_quran["query_id"] == sample_id, "doc_id"]:
    print(f"  {verse:<7} {quran.set_index('doc_id').loc[verse, 'text_en'][:110]}")
print("\nGround truth hadis (skor_teks, ✅ = terverifikasi):")
for _, h in qrels_hadith[qrels_hadith["query_id"] == sample_id].iterrows():
    mark = "✅" if h["terverifikasi"] else "⚠️"
    print(f"  {mark} {h['doc_id']:<15} skor {h['skor_teks']:.2f}  {hadith_text[h['doc_id']][:90]}")

ISLAMQA 4334: How to deal with a brother’s wife who has a bad attitude
https://islamqa.info/en/answers/4334

Kutipan ayat di jawaban: ['Fussilat 41:35', 'al-A’raaf 7:199']

Ground truth ayat:
  41:35   But none is granted it except those who are patient, and none is granted it except one having a great portion 
  7:199   Take what is given freely, enjoin what is good, and turn away from the ignorant.

Ground truth hadis (skor_teks, ✅ = terverifikasi):


In [10]:
# File yang sudah dibuat
for f in sorted(DATA_OUT.glob("*.csv")):
    print(f"{f.name:<22} {f.stat().st_size / 1e6:6.2f} MB")

hadith.csv              42.14 MB
islamqa.csv             10.01 MB
qrels_hadith.csv         0.06 MB
qrels_question.csv       0.00 MB
qrels_quran.csv          0.04 MB
quran.csv                2.37 MB
